# One extra layer on top of the 08b network (Kaggle, GPU)

**Step 3** (step 1 is notebook 08a, the data; step 2 is notebook 08b, the network with pawn inputs). This notebook takes the trained 08b network and adds **one extra layer** between the hidden neurons and the output:

- **08b:** 1152 inputs -> 1024 hidden neurons per side -> 1 output
- **08c:** 1152 inputs -> 1024 hidden neurons per side -> **N neurons** -> 1 output, with **N = 8, 16 and 24**

Everything else is the same as in 08b: the inputs (768 pieces + 384 pawn structure), the data (the cache of 08a + the Lichess games), the split into train, validation and test (same seed), the learning rate, the batch size and 6 epochs. So the losses of the three sizes can be compared directly with each other and with 08b.

- **Warm start:** the first neuron of the new layer gets the old output of 08b (scaled to the range 0..1), the other neurons start small and their output weight is 0. Every size therefore starts with exactly the 08b evaluation (for scores up to about +-1600 cp) and only learns what the extra layer adds.
- **The three sizes are trained one after the other** in the same session, each with its own checkpoint.
- **Engine:** the Rust engine already reads this format (NNU2 with an extra layer). The extra layer is computed with floats instead of the fast int16 code, so it costs speed. Measured on the laptop with random weights of the right size: 8 neurons -> 58% of the nodes per second of now, 16 -> 46%, 24 -> 35%.

## Before you start

1. **Notebook 08a** and **notebook 08b** have run with *Save & Run All*.
2. **Add Input:** *Your Work* -> notebook 08a (the cache) and notebook 08b (`nnue_weights.npz`), and the dataset with `lichess_games.csv`.

## Run

**Settings:** Accelerator **GPU T4 x2** (one GPU is used), no internet needed. Then **Save Version** -> **Save & Run All (Commit)**.

If the session stops before all sizes are done: add **the output of this notebook** as input as well and run it again. It continues where it stopped.

## Output (the *Output* tab of this notebook)

- `nnue_l2_8.bin`, `nnue_l2_16.bin`, `nnue_l2_24.bin`: the engine files (NNU2 with pawn inputs and an extra layer of 8, 16 or 24 neurons)
- `nnue_weights_l2_*.npz`: the weights, to continue training later
- `nnue_08c.log`, `nnue_08c.json` and `nnue_08c_curves.png`: the comparison of the three sizes with 08b
- `checkpoint_l2_*.pt` and `best_model_l2_*.pt`: to continue in a next run

In [ ]:
import json
import os
import shutil
import struct
import time

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from matplotlib import pyplot as plt
from torch import nn

## Settings

In [ ]:
notebook_start = time.time()

input_dir = "/kaggle/input"
out_dir = "/kaggle/working"

wdl_weight = 0.3
use_pawn_features = True
layer2_sizes = [8, 16, 24]

val_size = 2_000_000
test_size = 2_000_000
batch_size = 4096
gpu_data_fraction = 0.6

learning_rate = 0.0003
num_epochs = 6
session_hours = 11.5
reserve_minutes = 20
warm_range = 3.0

CP_SCALE = 0.00184104
PIECE_FEATURES = 768
PAWN_FEATURES = 384
NUM_FEATURES = PIECE_FEATURES + PAWN_FEATURES if use_pawn_features else PIECE_FEATURES

log_file = os.path.join(out_dir, "nnue_08c.log")
device = torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu")

print("device:", device, "| GPUs:", torch.cuda.device_count())


def run_files(num_layer2):
    return {
        "checkpoint": os.path.join(out_dir, f"checkpoint_l2_{num_layer2}.pt"),
        "best": os.path.join(out_dir, f"best_model_l2_{num_layer2}.pt"),
        "engine": os.path.join(out_dir, f"nnue_l2_{num_layer2}.bin"),
        "weights": os.path.join(out_dir, f"nnue_weights_l2_{num_layer2}.npz"),
    }

## Helpers and input files

In [ ]:
os.makedirs(out_dir, exist_ok=True)


def log(text):
    hours = (time.time() - notebook_start) / 3600
    line = f"[{time.strftime('%H:%M:%S')} | {hours:.2f} h] {text}"
    print(line, flush=True)
    with open(log_file, "a", encoding="utf-8") as f:
        f.write(line + "\n")


def synchronize():
    if device.type == "cuda":
        torch.cuda.synchronize()


def find_files(name):
    return sorted(os.path.join(folder, name) for folder, _, files in os.walk(input_dir) if name in files)


def input_listing():
    return {folder: sorted(files)[:5] for folder, _, files in os.walk(input_dir) if files}


assert device.type == "cuda", "No GPU: choose Accelerator GPU T4 x2 in the settings of the notebook"

previous_logs = find_files(os.path.basename(log_file))
resume_dir = os.path.dirname(previous_logs[0]) if previous_logs else None
if resume_dir is not None:
    for name in os.listdir(resume_dir):
        if os.path.isfile(os.path.join(resume_dir, name)):
            shutil.copy(os.path.join(resume_dir, name), out_dir)

log(f"New session on {device}, looking for the input files in {input_dir}...")
games_files = find_files("lichess_games.bin") + find_files("lichess_games.csv") + find_files("lichess_games.zip")
start_weights_files = find_files("nnue_weights.npz")
cache_files = find_files("done.json")
assert games_files, f"lichess_games.bin, lichess_games.csv or lichess_games.zip not found. Did you add the dataset with Add Input? Found: {input_listing()}"
assert start_weights_files, f"nnue_weights.npz (the output of notebook 08b) not found. Add Input -> Your Work -> notebook 08b. Found: {input_listing()}"
assert cache_files, f"The cache of notebook 08a (lichess_evals_cache/done.json) is not found. Run notebook 08a first and add its output with Add Input. Found: {input_listing()}"

games_file, start_weights_file, cache_dir = games_files[0], start_weights_files[0], os.path.dirname(cache_files[0])
with open(cache_files[0], encoding="utf-8") as f:
    cache_info = json.load(f)
log(f"Games: {games_file} ({os.path.getsize(games_file) / 1e9:.2f} GB)")
log(f"Start weights: {start_weights_file}")
log(f"Lichess evaluation cache: {cache_dir} ({cache_info['positions']:,} positions, depth >= {cache_info['min_depth']})")
if resume_dir is not None:
    log(f"Previous run of this notebook found in {resume_dir}: training continues from its checkpoints")

## Old data: the cache of notebook 08a

About 143 million quiet positions from the Lichess evaluation database (Stockfish depth >= 25, the best move is not a capture or promotion), already packed by notebook 08a: 32 bytes per board, the side to move and the label for white. `parse_fen` is used for the test positions at the end.

In [ ]:
PIECE_CODES = {"P": 1, "N": 2, "B": 3, "R": 4, "Q": 5, "K": 6, "p": 7, "n": 8, "b": 9, "r": 10, "q": 11, "k": 12}
DIGITS = "12345678"
CACHE_NAMES = ["packed", "side", "labels"]


def parse_fen(fen):
    board_part, color = fen.split(" ")[:2]
    row = np.zeros(64, dtype=np.uint8)
    index = 0
    for char in board_part:
        if char == "/":
            continue
        if char in DIGITS:
            index += int(char)
        else:
            row[index ^ 56] = PIECE_CODES[char]
            index += 1
    return row, color == "w"


def open_cache():
    return tuple(np.load(os.path.join(cache_dir, f"{name}.npy"), mmap_mode="r") for name in CACHE_NAMES)

## New data: positions from Lichess games (notebook 07)

The games file can be in two formats with the same content:

- `lichess_games.bin` (the output of notebook 07): 36 bytes per position: the board (32 bytes, the same packing as the cache), the side to move, the Stockfish score in centipawns for white (mate = ±10000) and the result of the game (0 black won, 1 draw, 2 white won)
- `lichess_games.csv` (also inside a zip file): one position per line with the columns `board` (the first part of a FEN), `turn` (`w` or `b`), `score` (centipawns for white, mate = ±10000) and `result` (1 white won, 0.5 draw, 0 black won)

The csv file is read in parts of 2 million lines. Each part of FEN boards is packed in one go: every digit becomes that many empty squares (`1`), so every board is exactly 64 characters, and a lookup table turns the characters into piece codes.

The target is `(1 - wdl_weight) * score + wdl_weight * result`, both as a number between -1 and 1.

In [ ]:
GAME_RECORD = np.dtype([("board", "u1", 32), ("white", "u1"), ("score", "<i2"), ("result", "u1")])
GAME_COLUMNS = {"board": str, "turn": str, "score": np.int16, "result": np.float32}
FEN_CODES = np.zeros(256, dtype=np.uint8)
for char, piece_code in PIECE_CODES.items():
    FEN_CODES[ord(char)] = piece_code
FEN_SQUARES = np.arange(64) ^ 56


def game_labels(score, result):
    score_labels = 2 / (1 + np.exp(-0.00368208 * score.astype(np.float32))) - 1
    result_labels = 2 * result.astype(np.float32) - 1
    return ((1 - wdl_weight) * score_labels + wdl_weight * result_labels).astype(np.float16)


def pack_boards(boards):
    text = "".join(boards)
    for digit in "2345678":
        text = text.replace(digit, "1" * int(digit))
    chars = np.frombuffer(text.replace("/", "").encode("ascii"), dtype=np.uint8)
    assert len(chars) == 64 * len(boards), "a board in the games file is not a valid FEN board"
    rows = FEN_CODES[chars.reshape(-1, 64)][:, FEN_SQUARES]
    return rows[:, 0::2] | (rows[:, 1::2] << 4)


def load_games(path, limit=None):
    if path.endswith(".bin"):
        records = np.memmap(path, dtype=GAME_RECORD, mode="r")[:limit]
        return np.ascontiguousarray(records["board"]), np.array(records["white"]), game_labels(records["score"], records["result"] / 2)
    parts = []
    for chunk in pd.read_csv(path, dtype=GAME_COLUMNS, chunksize=2_000_000, nrows=limit):
        packed = pack_boards(chunk["board"].tolist())
        white = (chunk["turn"] == "w").to_numpy(np.uint8)
        parts.append((packed, white, game_labels(chunk["score"].to_numpy(), chunk["result"].to_numpy())))
    return tuple(np.concatenate(arrays) for arrays in zip(*parts))

## Features: 768 piece inputs + 384 pawn structure inputs per perspective

Each perspective sees the board from its own side: for black the board is flipped, so "own" and "enemy" always mean the same thing.

The pawn structure is computed on the GPU for a whole batch at once, on an 8x8 grid per board (rank, file):

- **passed:** no enemy pawn in front of it on the same or an adjacent file
- **isolated:** no own pawn on an adjacent file
- **doubled:** another own pawn on the same file

The input number is `768 + kind * 128 + (0 own, 64 enemy) + square`, exactly like `pawn_feature` in the engine.

In [ ]:
PIECE_TYPES = torch.tensor([0, 0, 1, 2, 3, 4, 5, 0, 1, 2, 3, 4, 5], device=device)
PIECE_COLORS = torch.tensor([0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 1, 1], device=device)
SQUARES = torch.arange(64, device=device)
PAWN_KINDS = torch.arange(3, device=device).view(1, 3, 1) * 128


def unpack(data_packed):
    low = data_packed & 15
    high = data_packed >> 4
    return torch.stack([low, high], dim=2).reshape(-1, 64).long()


def north(x):
    return (F.pad(x[:, :-1, :], (0, 0, 1, 0)).cumsum(dim=1) > 0).int()


def south(x):
    return north(x.flip(1)).flip(1)


def adjacent(x):
    return F.pad(x[:, :, :-1], (1, 0)) | F.pad(x[:, :, 1:], (0, 1))


def pawn_structure(board):
    pawns = [(board == 1).int().view(-1, 8, 8), (board == 7).int().view(-1, 8, 8)]
    structure = []
    for color in (0, 1):
        own, enemy = pawns[color], pawns[1 - color]
        blockers = south(enemy) if color == 0 else north(enemy)
        files = own | north(own) | south(own)
        passed = own & (1 - (blockers | adjacent(blockers)))
        isolated = own & (1 - adjacent(files))
        doubled = own & (north(own) | south(own))
        structure.append(torch.stack([passed, isolated, doubled], dim=1).view(-1, 3, 64))
    return structure


def perspective_features(board, structure, perspective):
    flipped = SQUARES ^ (56 * perspective)
    own = (PIECE_COLORS[board] != perspective).long() * 384
    pieces = (own + PIECE_TYPES[board] * 64 + flipped).masked_fill(board == 0, NUM_FEATURES)
    if not use_pawn_features:
        return pieces.sort(dim=1).values[:, :32]
    parts = [pieces]
    for color in (0, 1):
        index = PIECE_FEATURES + (0 if color == perspective else 64) + PAWN_KINDS + flipped.view(1, 1, 64)
        parts.append(torch.where(structure[color].bool(), index, NUM_FEATURES).view(-1, 192))
    x = torch.cat(parts, dim=1).sort(dim=1).values
    return x[:, :int((x < NUM_FEATURES).sum(dim=1).max())]


def features(board, white):
    structure = pawn_structure(board) if use_pawn_features else None
    x_white = perspective_features(board, structure, 0)
    x_black = perspective_features(board, structure, 1)
    width = max(x_white.shape[1], x_black.shape[1])
    x_white = F.pad(x_white, (0, width - x_white.shape[1]), value=NUM_FEATURES)
    x_black = F.pad(x_black, (0, width - x_black.shape[1]), value=NUM_FEATURES)
    white = white.bool().unsqueeze(1)
    return torch.where(white, x_white, x_black), torch.where(white, x_black, x_white)

## Model: the 08b network plus one extra layer

The inputs are added up per perspective (`EmbeddingBag`) and clipped between 0 and 1, exactly like 08b. New is the layer `hidden`: 2048 values (side to move first) -> N neurons, clipped between 0 and 1, and then `output`: N -> 1. During training the output goes through `tanh`, like the labels; the engine uses the value before `tanh`.

**Warm start** in `build_model`, with `y` the output of 08b and `r = warm_range = 3`:

- neuron 0 of the new layer: `clamp(0.5 + y / (2r), 0, 1)`
- output: `2r * neuron0 - r`, which is exactly `y` as long as `-3 <= y <= 3` (about +-1600 cp)
- the other neurons: small random weights and bias 0.5, so they are not stuck at 0 or 1, and output weight 0, so at the start they change nothing

In [ ]:
class NNUE(nn.Module):

    def __init__(self, num_hidden, num_layer2):
        super().__init__()
        self.feature_transformer = nn.EmbeddingBag(NUM_FEATURES + 1, num_hidden, mode="sum", padding_idx=NUM_FEATURES)
        self.feature_bias = nn.Parameter(torch.zeros(num_hidden))
        self.hidden = nn.Linear(in_features=2 * num_hidden, out_features=num_layer2)
        self.output = nn.Linear(in_features=num_layer2, out_features=1)

    def forward(self, x_us, x_them):
        us = torch.clamp(self.feature_transformer(x_us) + self.feature_bias, 0, 1)
        them = torch.clamp(self.feature_transformer(x_them) + self.feature_bias, 0, 1)
        x = torch.cat([us, them], dim=1)
        x = torch.clamp(self.hidden(x), 0, 1)
        return self.output(x)[:, 0]


def build_model(start, num_layer2):
    weight = start["feature_transformer.weight"]
    assert weight.shape[1] == NUM_FEATURES, "the start weights are not the 08b weights with pawn inputs"
    model = NNUE(num_hidden=weight.shape[0], num_layer2=num_layer2)
    with torch.no_grad():
        model.feature_transformer.weight.zero_()
        model.feature_transformer.weight[:weight.shape[1]] = torch.from_numpy(weight.T.copy())
        model.feature_bias.copy_(torch.from_numpy(start["feature_transformer.bias"]))
        model.hidden.weight.normal_(0, 0.01)
        model.hidden.bias.fill_(0.5)
        model.hidden.weight[0] = torch.from_numpy(start["output.weight"][0]) / (2 * warm_range)
        model.hidden.bias[0] = 0.5 + float(start["output.bias"][0]) / (2 * warm_range)
        model.output.weight.zero_()
        model.output.weight[0, 0] = 2 * warm_range
        model.output.bias.fill_(-warm_range)
    return model.to(device)

## Engine file (nnue.bin, format NNU2 with pawn inputs and an extra layer) and a Python version of the engine evaluation

The NNU2 header has a field for the size of the extra layer (`layer2`, 0 in 08b). After the weights of the inputs come the weights of the extra layer (N rows of 2048), its N biases, the N output weights and the output bias. The engine (`evaluate` in `nnue.rs`) computes for every neuron `clamp(bias + weights . inputs, 0, 1)` and adds them up with the output weights.

Before this notebook was made, `write_nnue`, `read_nnue` and `reference_eval` were compared with the Rust engine (the `eval` command) on random networks of this size with 0, 8, 16 and 24 extra neurons, 10 positions each: the largest difference was 0.0016 (about 1 cp, from the int16 rounding in the engine).

In [ ]:
FILE_A = 0x0101010101010101
FILE_H = FILE_A << 7
FULL = (1 << 64) - 1


def write_nnue(net, path):
    num_features, num_hidden = net["feature_weights"].shape
    num_layer2 = len(net["hidden_bias"])
    with open(path, "wb") as f:
        f.write(b"NNU2")
        f.write(struct.pack("<5I", 1, 0, int(num_features > PIECE_FEATURES), num_hidden, num_layer2))
        f.write(struct.pack("<64I", *([0] * 64)))
        for part in (net["feature_weights"], net["feature_bias"], np.zeros(num_features), net["hidden_weight"], net["hidden_bias"], net["output_weight"], net["output_bias"]):
            f.write(np.asarray(part, dtype="<f4").tobytes())


def read_nnue(path):
    with open(path, "rb") as f:
        data = f.read()
    assert data[:4] == b"NNU2", "not an NNU2 file"
    num_buckets, mirror, pawns, num_hidden, num_layer2 = struct.unpack("<5I", data[4:24])
    assert (num_buckets, mirror) == (1, 0), "unexpected NNU2 header"
    num_features = PIECE_FEATURES + PAWN_FEATURES * pawns
    num_outputs = num_layer2 if num_layer2 else 2 * num_hidden
    values = np.frombuffer(data, dtype="<f4", offset=24 + 64 * 4)
    sizes = [num_features * num_hidden, num_hidden, num_features, num_layer2 * 2 * num_hidden, num_layer2, num_outputs, 1]
    assert sum(sizes) == len(values), "unexpected file size"
    parts = np.split(values, np.cumsum(sizes)[:-1])
    return {
        "feature_weights": parts[0].reshape(num_features, num_hidden),
        "feature_bias": parts[1],
        "hidden_weight": parts[3].reshape(num_layer2, 2 * num_hidden),
        "hidden_bias": parts[4],
        "output_weight": parts[5].reshape(1, num_outputs),
        "output_bias": parts[6],
    }


def north_span(squares):
    span = (squares << 8) & FULL
    span |= (span << 8) & FULL
    span |= (span << 16) & FULL
    return span | ((span << 32) & FULL)


def south_span(squares):
    span = squares >> 8
    span |= span >> 8
    span |= span >> 16
    return span | (span >> 32)


def adjacent_files(squares):
    return ((squares << 1) & ~FILE_A & FULL) | ((squares >> 1) & ~FILE_H)


def reference_pawn_structure(row):
    pawns = [0, 0]
    for square in range(64):
        if row[square] == 1:
            pawns[0] |= 1 << square
        elif row[square] == 7:
            pawns[1] |= 1 << square
    structure = []
    for color in (0, 1):
        own, enemy = pawns[color], pawns[1 - color]
        blockers = south_span(enemy) if color == 0 else north_span(enemy)
        files = own | north_span(own) | south_span(own)
        passed = own & ~(blockers | adjacent_files(blockers))
        isolated = own & ~adjacent_files(files)
        doubled = own & (north_span(own) | south_span(own))
        structure.append([passed, isolated, doubled])
    return structure


def reference_features(row, perspective):
    flip = 56 * perspective
    active = []
    for square in range(64):
        code = int(row[square])
        if code == 0:
            continue
        own = 0 if (code <= 6) == (perspective == 0) else 384
        active.append(own + (code - 1) % 6 * 64 + (square ^ flip))
    if use_pawn_features:
        for color, kinds in enumerate(reference_pawn_structure(row)):
            own = 0 if color == perspective else 64
            for kind, squares in enumerate(kinds):
                for square in range(64):
                    if squares >> square & 1:
                        active.append(PIECE_FEATURES + kind * 128 + own + (square ^ flip))
    return active


def reference_eval(net, row, white):
    us, them = (0, 1) if white else (1, 0)
    hidden = []
    for perspective in (0, 1):
        active = reference_features(row, perspective)
        hidden.append(np.clip(net["feature_bias"] + net["feature_weights"][active].sum(axis=0), 0, 1))
    x = np.concatenate([hidden[us], hidden[them]])
    if len(net["hidden_bias"]):
        x = np.clip(net["hidden_weight"] @ x + net["hidden_bias"], 0, 1)
    return float((net["output_weight"] @ x + net["output_bias"])[0])

In [ ]:
def model_net(model):
    weights = {name: value.detach().cpu().numpy() for name, value in model.state_dict().items()}
    return {
        "feature_weights": weights["feature_transformer.weight"][:NUM_FEATURES],
        "feature_bias": weights["feature_bias"],
        "hidden_weight": weights["hidden.weight"],
        "hidden_bias": weights["hidden.bias"],
        "output_weight": weights["output.weight"],
        "output_bias": weights["output.bias"],
    }


def start_net(start):
    weight = start["feature_transformer.weight"].T
    return {
        "feature_weights": weight,
        "feature_bias": start["feature_transformer.bias"],
        "hidden_weight": np.zeros((0, 2 * weight.shape[1]), dtype=np.float32),
        "hidden_bias": np.zeros(0, dtype=np.float32),
        "output_weight": start["output.weight"],
        "output_bias": start["output.bias"],
    }


def save_weights(model, path):
    net = model_net(model)
    np.savez(
        path,
        num_hidden=np.array(net["feature_bias"].shape[0]),
        num_layer2=np.array(net["hidden_bias"].shape[0]),
        **{
            "feature_transformer.weight": net["feature_weights"].T,
            "feature_transformer.bias": net["feature_bias"],
            "hidden.weight": net["hidden_weight"],
            "hidden.bias": net["hidden_bias"],
            "output.weight": net["output_weight"],
            "output.bias": net["output_bias"],
        },
    )


def export_model(model, path):
    write_nnue(model_net(model), path)


def predict(model, row, white):
    board = torch.from_numpy(np.asarray(row)).long().unsqueeze(0).to(device)
    data_white = torch.tensor([white], device=device)
    model.eval()
    with torch.no_grad():
        x_us, x_them = features(board, data_white)
        return model(x_us, x_them).item()

## Training functions

In [ ]:
loss_fn = nn.MSELoss()


def batches(data, indices, shuffle):
    data_packed, data_side, data_labels = data
    if shuffle:
        indices = indices[torch.randperm(len(indices)).to(indices.device)]
    for start in range(0, len(indices), batch_size):
        batch = indices[start:start + batch_size].long()
        board = unpack(data_packed[batch].to(device, non_blocking=True))
        white = data_side[batch].to(device, non_blocking=True).bool()
        x_us, x_them = features(board, white)
        labels = data_labels[batch].to(device, non_blocking=True).float()
        targets = torch.where(white, labels, -labels)
        yield x_us, x_them, targets


def train_one_epoch(model, optimizer, data, indices):
    model.train()
    total_loss, num_batches = 0., 0

    for x_us, x_them, targets in batches(data, indices, shuffle=True):
        preds = torch.tanh(model(x_us, x_them))
        loss = loss_fn(preds, targets)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.detach()
        num_batches += 1

    return (total_loss / num_batches).item()


def eval_model(model, data, indices):
    model.eval()
    total_loss, num_batches = 0., 0
    total_abs_error, num_preds = 0., 0

    with torch.no_grad():
        for x_us, x_them, targets in batches(data, indices, shuffle=False):
            preds = torch.tanh(model(x_us, x_them))

            total_loss += loss_fn(preds, targets)
            num_batches += 1

            total_abs_error += (preds - targets).abs().sum()
            num_preds += targets.shape[0]

    return (total_loss / num_batches).item(), (total_abs_error / num_preds).item()


def measure_speed(learning_rate, data, indices, num_layer2, num_batches=300, warmup=50):
    model = build_model(start_weights, num_layer2)
    optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)
    model.train()

    for i, (x_us, x_them, targets) in enumerate(batches(data, indices, shuffle=True)):
        if i == warmup:
            synchronize()
            start = time.time()
        if i == num_batches:
            break
        loss = loss_fn(torch.tanh(model(x_us, x_them)), targets)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

    synchronize()
    return (num_batches - warmup) * batch_size / (time.time() - start)

## Quick check of the whole pipeline

Fails within minutes if something is wrong:

- every size starts with exactly the 08b evaluation (warm start)
- the pawn and piece inputs on the GPU are the same as in the Python copy of the engine
- training works, and the exported engine file with the extra layer gives the same evaluation as the model
- the games file and the cache of notebook 08a have exactly one king per side in every position

In [ ]:
log("Quick check of the whole pipeline on a small sample...")
start_weights = dict(np.load(start_weights_file))
log(f"08b start weights: {start_weights['feature_transformer.weight'].shape[1]} inputs -> {start_weights['feature_transformer.weight'].shape[0]} hidden -> 1 output")

sample = tuple(torch.from_numpy(array).to(device) for array in load_games(games_file, limit=20_000))
log(f"First {len(sample[2]):,} positions of {os.path.basename(games_file)}: white to move in {sample[1].float().mean().item():.0%}, mean target for white {sample[2].float().mean().item():+.3f}")
sample_idx = torch.arange(len(sample[2]), device=device, dtype=torch.int32)
board = unpack(sample[0][:1000])
assert bool(((board == 6).sum(dim=1) == 1).all()) and bool(((board == 12).sum(dim=1) == 1).all()), "every position needs exactly one king per side"

check_model = build_model(start_weights, layer2_sizes[-1])
start_reference = start_net(start_weights)
feature_errors, start_error = 0, 0.
for i in range(300):
    row = unpack(sample[0][i:i + 1])[0].cpu().numpy()
    white = bool(sample[1][i])
    x_us, x_them = features(torch.from_numpy(row).long().unsqueeze(0).to(device), torch.tensor([white], device=device))
    us = 0 if white else 1
    for x, perspective in [(x_us, us), (x_them, 1 - us)]:
        if sorted(index for index in x[0].tolist() if index < NUM_FEATURES) != sorted(reference_features(row, perspective)):
            feature_errors += 1
    expected = np.clip(reference_eval(start_reference, row, white), -warm_range, warm_range)
    start_error = max(start_error, abs(predict(check_model, row, white) - expected))
log(f"Feature mismatches: {feature_errors}, largest difference between the start model and 08b: {start_error:.6f}")
assert feature_errors == 0, "the features differ from the engine"
assert start_error < 1e-4, "the start model does not give the 08b evaluation"

check_optimizer = torch.optim.Adam(check_model.parameters(), lr=1e-3)
train_loss = train_one_epoch(check_model, check_optimizer, sample, sample_idx)
val_loss, val_mae = eval_model(check_model, sample, sample_idx)
log(f"One epoch on the sample: train loss {train_loss:.4f}, val loss {val_loss:.4f}, val MAE {val_mae:.4f}")
log(f"Speed with {layer2_sizes[-1]} extra neurons: {measure_speed(1e-3, sample, sample_idx, layer2_sizes[-1], num_batches=4, warmup=1):,.0f} positions/s")

with torch.no_grad():
    for param in check_model.parameters():
        param.add_(0.01 * torch.randn_like(param))
check_path = os.path.join(out_dir, "check.bin")
export_model(check_model, check_path)
check_net = read_nnue(check_path)
os.remove(check_path)
export_error = 0.
for i in range(300):
    row = unpack(sample[0][i:i + 1])[0].cpu().numpy()
    white = bool(sample[1][i])
    export_error = max(export_error, abs(predict(check_model, row, white) - reference_eval(check_net, row, white)))
log(f"Largest difference between the model and the engine file: {export_error:.6f}")
assert export_error < 1e-3, "the engine file does not match the model"

cache = open_cache()
assert len(cache[2]) == cache_info["positions"], "the cache of notebook 08a is not complete"
cache_board = unpack(torch.from_numpy(np.array(cache[0][:1000])).to(device))
assert bool(((cache_board == 6).sum(dim=1) == 1).all()) and bool(((cache_board == 12).sum(dim=1) == 1).all()), "every cached position needs exactly one king per side"
log("Quick check passed")

## Data

The cache of notebook 08a and the games are put together in one dataset. Everything is kept on the GPU if it fits (about 7 GB), so training does not wait for the CPU.

The split into train, validation and test uses a fixed seed, so it is the same in every session.

In [ ]:
log("Loading the cache and the games...")
games = load_games(games_file)
num_lichess, num_games = len(cache[2]), len(games[2])
num_positions = num_lichess + num_games
arrays = tuple(np.concatenate([cached, part]) for cached, part in zip(cache, games))
del cache, games, sample, cache_board

data_gb = sum(array.nbytes for array in arrays) / 1e9
gpu_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
data_device = device if data_gb < gpu_data_fraction * gpu_gb else torch.device("cpu")
data = tuple(torch.from_numpy(array).to(data_device) for array in arrays)
del arrays
log(f"Data: {num_positions:,} positions ({num_lichess:,} Lichess evaluations, {num_games:,} from games), {data_gb:.1f} GB, kept on {data_device}")

perm = torch.randperm(num_positions, generator=torch.Generator().manual_seed(42)).int().to(data_device)
val_idx = perm[:val_size]
test_idx = perm[val_size:val_size + test_size]
train_idx = perm[val_size + test_size:]
del perm
log(f"Train: {len(train_idx):,}, val: {len(val_idx):,}, test: {len(test_idx):,}")
log(f"GPU memory in use: {torch.cuda.memory_allocated() / 1e9:.1f} GB of {gpu_gb:.1f} GB")

## Training: the three sizes one after the other (continues from the checkpoints if there are any)

For every size: the warm start from 08b, Adam with the same cosine learning rate schedule as 08b over `num_epochs` epochs. After every epoch the checkpoint of that size is saved. When the validation loss of that size is the best so far, its engine file and weights are saved as well.

The first line is the validation loss of 08b itself (the warm start), the number every size has to beat.

In [ ]:
deadline = notebook_start + session_hours * 3600 - reserve_minutes * 60
speed = measure_speed(learning_rate, data, train_idx, layer2_sizes[-1])
epoch_seconds = 1.1 * len(train_idx) / speed
log(f"Speed with {layer2_sizes[-1]} extra neurons: {speed:,.0f} positions/s, about {epoch_seconds / 60:.0f} min per epoch, {num_epochs} epochs for each size {layer2_sizes}")

start_val_loss, start_val_mae = eval_model(build_model(start_weights, layer2_sizes[0]), data, val_idx)
log(f"Before training (the 08b network): val loss {start_val_loss:.4f}, val MAE {start_val_mae:.4f}")

results = {}
for num_layer2 in layer2_sizes:
    files = run_files(num_layer2)
    model = build_model(start_weights, num_layer2)
    optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=num_epochs, eta_min=learning_rate / 100)
    history = {"train_loss": [], "val_loss": [], "val_mae": [], "learning_rate": []}
    best_val_loss = float("inf")

    if os.path.exists(files["checkpoint"]):
        checkpoint = torch.load(files["checkpoint"], map_location=device, weights_only=False)
        model.load_state_dict(checkpoint["model"])
        optimizer.load_state_dict(checkpoint["optimizer"])
        scheduler.load_state_dict(checkpoint["scheduler"])
        history = checkpoint["history"]
        best_val_loss = checkpoint["best_val_loss"]
        log(f"[{num_layer2} neurons] Continuing from the checkpoint after epoch {len(history['train_loss'])}")
    else:
        log(f"[{num_layer2} neurons] New extra layer 2048 -> {num_layer2} -> 1, starting from the 08b network")

    for epoch in range(len(history["train_loss"]), num_epochs):
        if time.time() + epoch_seconds > deadline:
            log("Not enough time left in this Kaggle session for another epoch: add the output of this notebook as input and run it again to continue")
            break

        current_lr = optimizer.param_groups[0]["lr"]
        train_loss = train_one_epoch(model, optimizer, data, train_idx)
        val_loss, val_mae = eval_model(model, data, val_idx)
        scheduler.step()

        history["train_loss"].append(train_loss)
        history["val_loss"].append(val_loss)
        history["val_mae"].append(val_mae)
        history["learning_rate"].append(current_lr)
        log(f"[{num_layer2} neurons] Epoch {epoch + 1}/{num_epochs}: train loss {train_loss:.4f}, val loss {val_loss:.4f}, val MAE {val_mae:.4f}, lr {current_lr:.6f}")

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            torch.save(model.state_dict(), files["best"])
            save_weights(model, files["weights"])
            export_model(model, files["engine"])
            log(f"[{num_layer2} neurons] Best so far: saved {os.path.basename(files['engine'])}")
        torch.save({"model": model.state_dict(), "optimizer": optimizer.state_dict(), "scheduler": scheduler.state_dict(), "history": history, "best_val_loss": best_val_loss}, files["checkpoint"])

    results[num_layer2] = {"history": history, "best_val_loss": best_val_loss, "finished": len(history["train_loss"]) == num_epochs}
    del model, optimizer, scheduler
    if not results[num_layer2]["finished"]:
        break

finished = len(results) == len(layer2_sizes) and all(result["finished"] for result in results.values())
log("Training finished for all sizes" if finished else f"Training paused, finished sizes: {[size for size, result in results.items() if result['finished']]}")

## Test

The best network of every size on the test positions, compared with 08b on the same positions.

In [ ]:
start_test_loss, start_test_mae = eval_model(build_model(start_weights, layer2_sizes[0]), data, test_idx)
log(f"08b without an extra layer: test loss {start_test_loss:.4f}, MAE {start_test_mae:.4f}")

trained_sizes = [size for size in results if os.path.exists(run_files(size)["best"])]
test_results = {}
for num_layer2 in trained_sizes:
    model = build_model(start_weights, num_layer2)
    model.load_state_dict(torch.load(run_files(num_layer2)["best"], map_location=device))
    test_loss, test_mae = eval_model(model, data, test_idx)
    test_results[num_layer2] = {"test_loss": test_loss, "test_mae": test_mae}
    log(f"[{num_layer2} neurons] test loss {test_loss:.4f}, MAE {test_mae:.4f} ({(test_loss / start_test_loss - 1) * 100:+.1f}% compared with 08b)")

if not finished:
    log("Not all sizes are finished. To continue: Add Input -> the output of this notebook, then Save & Run All again")

## Sanity check

A few test positions for every size, from the model and from its engine file. The two numbers must be the same.

In [ ]:
test_fens = {
    "start position": "rnbqkbnr/pppppppp/8/8/8/8/PPPPPPPP/RNBQKBNR w KQkq - 0 1",
    "white a bishop down, white to move": "rnbqkbnr/pppppppp/8/8/8/8/PPPPPPPP/RNBQK1NR w KQkq - 0 1",
    "white a queen down, white to move": "rnbqkbnr/pppppppp/8/8/8/8/PPPPPPPP/RNB1KBNR w KQkq - 0 1",
    "rook against 3 pawns, white to move (about +200)": "1nbqkbnr/pppppppp/8/8/8/8/3PPPPP/RNBQKBNR w KQk - 0 1",
    "lost game 1, black a bishop down (Stockfish -639)": "2r3k1/5p1p/bq2p1p1/pN1pP3/P2P2n1/RP1Q1NP1/5PBP/6K1 b - - 0 22",
    "lost game 2, black to move (Stockfish -387)": "2krr3/1pp2p1p/p1nb3q/3p4/Q1PP2pN/3bP1P1/P2N1PBP/R2R2K1 b - - 0 16",
}

sanity = {}
for num_layer2 in trained_sizes:
    files = run_files(num_layer2)
    model = build_model(start_weights, num_layer2)
    model.load_state_dict(torch.load(files["best"], map_location=device))
    engine_net = read_nnue(files["engine"])
    sanity[num_layer2] = {}
    for name, fen in test_fens.items():
        row, white = parse_fen(fen)
        raw, engine_raw = predict(model, row, white), reference_eval(engine_net, row, white)
        sanity[num_layer2][name] = raw / CP_SCALE
        log(f"[{num_layer2} neurons] {name}: {raw / CP_SCALE:+.0f} cp for the side to move (engine file: {engine_raw / CP_SCALE:+.0f} cp)")

params = {
    "num_hidden": int(start_weights["feature_transformer.bias"].shape[0]),
    "num_features": NUM_FEATURES,
    "layer2_sizes": layer2_sizes,
    "activation": "clipped relu",
    "start": "08b",
    "warm_range": warm_range,
    "learning_rate": learning_rate,
    "wdl_weight": wdl_weight,
    "num_lichess": num_lichess,
    "num_games": num_games,
    "min_depth": cache_info["min_depth"],
    "num_epochs": num_epochs,
    "positions_per_second": speed,
}
summary = {
    "params": params,
    "start_val_loss": start_val_loss,
    "start_test_loss": start_test_loss,
    "start_test_mae": start_test_mae,
    "results": {str(size): {**results[size], **test_results.get(size, {})} for size in results},
    "sanity": {str(size): values for size, values in sanity.items()},
}
with open(os.path.join(out_dir, "nnue_08c.json"), "w", encoding="utf-8") as f:
    json.dump(summary, f, indent=2)
log("Saved nnue_08c.json")

## Plots

In [ ]:
plt.figure(figsize=(12, 4))

plt.subplot(1, 2, 1)
for num_layer2, result in results.items():
    epochs = range(1, len(result["history"]["val_loss"]) + 1)
    plt.plot(epochs, result["history"]["val_loss"], marker="o", label=f"{num_layer2} neurons")
plt.axhline(start_val_loss, color="gray", linestyle="--", label="08b (no extra layer)")
plt.xlabel("Epoch")
plt.ylabel("Validation MSE")
plt.legend()

plt.subplot(1, 2, 2)
plt.bar([str(size) for size in test_results], [result["test_loss"] for result in test_results.values()])
plt.axhline(start_test_loss, color="gray", linestyle="--", label="08b (no extra layer)")
plt.xlabel("Neurons in the extra layer")
plt.ylabel("Test MSE")
plt.legend()

plt.savefig(os.path.join(out_dir, "nnue_08c_curves.png"), bbox_inches="tight")
plt.show()

In [ ]:
log(f"Done. Download nnue_l2_8.bin, nnue_l2_16.bin and nnue_l2_24.bin from the Output tab of this notebook. Files: {sorted(os.listdir(out_dir))}")